# LoRA: Low-Rank Decomposition Math, PEFT Library, and Training on a Real Dataset

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/research_5)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch transformers peft bitsandbytes

In [ ]:
import torch
import math

def lora_parameter_count(d: int, k: int, r: int) -> tuple:
    """Parameters in LoRA vs full fine-tuning."""
    full_ft = d * k
    lora = d * r + r * k  # B + A
    return full_ft, lora, full_ft / lora

# Typical transformer dimensions
configs = [
    ("Attention Q/K (7B, head_dim=128)", 4096, 4096, 8),
    ("Attention Q/K (7B, head_dim=128)", 4096, 4096, 16),
    ("MLP fc1 (7B)", 4096, 11008, 8),
    ("Attention (1B, small model)", 2048, 2048, 8),
]

print(f"{'Layer':<35} {'Full FT':>10} {'LoRA':>8} {'Reduction':>10}")
print("-" * 70)
for name, d, k, r in configs:
    full, lora, ratio = lora_parameter_count(d, k, r)
    print(f"{name:<35} {full:>10,} {lora:>8,} {ratio:>9.0f}×")

In [ ]:
import torch
import torch.nn as nn
import math

class LoRALayer(nn.Module):
    """
    LoRA adapter for a linear layer.
    Replaces W₀x with W₀x + (B @ A)x during the forward pass.
    """
    def __init__(self, in_features: int, out_features: int, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank  # LoRA scaling factor

        # Pretrained weight (frozen)
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.weight.requires_grad = False
        self.bias.requires_grad = False

        # LoRA matrices (trainable)
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        # Initialize A with kaiming_uniform (standard), B with zeros
        # Zero-init of B ensures ΔW = 0 at the start — safe to start training
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Frozen base output
        base_out = nn.functional.linear(x, self.weight, self.bias)
        # LoRA update: x @ A^T @ B^T * scaling
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

    @property
    def effective_weight(self) -> torch.Tensor:
        """Merge LoRA into weight for deployment (no runtime overhead)."""
        return self.weight + (self.lora_B @ self.lora_A) * self.scaling


# Test
torch.manual_seed(42)
layer = LoRALayer(in_features=512, out_features=512, rank=8, alpha=16)
x = torch.randn(4, 512)
out = layer(x)

total_params = sum(p.numel() for p in layer.parameters())
trainable_params = sum(p.numel() for p in layer.parameters() if p.requires_grad)

print(f"Output shape: {out.shape}")
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Frozen params:    {total_params - trainable_params:,}")
print(f"LoRA matrices: A={layer.lora_A.shape}, B={layer.lora_B.shape}")

In [ ]:
import torch
import torch.nn as nn
import math

def demonstrate_zero_init():
    """Show that zero-init of B ensures LoRA starts as identity."""
    torch.manual_seed(0)

    layer_zero = LoRALayer(256, 256, rank=4)  # B initialized to zeros
    layer_random = LoRALayer(256, 256, rank=4)
    # Override B with random values to show the difference
    nn.init.kaiming_uniform_(layer_random.lora_B, a=math.sqrt(5))

    x = torch.randn(2, 256)

    base_out = nn.functional.linear(x, layer_zero.weight, layer_zero.bias)
    zero_init_out = layer_zero(x)
    random_init_out = layer_random(x)

    diff_zero = (zero_init_out - base_out).abs().max().item()
    diff_random = (random_init_out - base_out).abs().max().item()

    print(f"Zero-init B: max difference from base model = {diff_zero:.8f}")
    print(f"Random B:    max difference from base model = {diff_random:.4f}")

demonstrate_zero_init()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Load a small model for demonstration
model_name = "facebook/opt-125m"  # 125M params, CPU-friendly
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Original parameters: {sum(p.numel() for p in model.parameters()):,}")

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # rank
    lora_alpha=16,          # scaling factor
    lora_dropout=0.1,       # dropout on LoRA layers
    target_modules=["q_proj", "v_proj"],  # which layers to apply LoRA to
    bias="none",            # don't add LoRA to bias terms
)

# Apply LoRA to model
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch

model_name = "facebook/opt-125m"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Apply LoRA
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM,
)
peft_model = get_peft_model(model, lora_config)

# Tiny dataset for demonstration
samples = [
    "The transformer architecture uses self-attention to process sequences.",
    "Gradient descent optimizes model parameters by following the negative gradient.",
    "Backpropagation computes gradients using the chain rule of calculus.",
    "LoRA enables efficient fine-tuning with low-rank weight updates.",
    "FAISS enables fast approximate nearest neighbor search for embeddings.",
] * 20  # repeat for more steps

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64, padding="max_length")

dataset = Dataset.from_dict({"text": samples})
tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized = tokenized.map(lambda x: {"labels": x["input_ids"]})

training_args = TrainingArguments(
    output_dir="/tmp/lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized,
)

trainer.train()
print("LoRA fine-tuning complete.")

In [ ]:
from peft import PeftModel

# Merge LoRA into base weights
merged_model = peft_model.merge_and_unload()

print(f"Original model class: {type(peft_model)}")
print(f"Merged model class:   {type(merged_model)}")
print(f"Merged model params:  {sum(p.numel() for p in merged_model.parameters()):,}")

# Verify output unchanged
import torch
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer("The transformer architecture", return_tensors="pt")

with torch.no_grad():
    peft_out = peft_model.generate(**inputs, max_new_tokens=5, do_sample=False)
    merged_out = merged_model.generate(**inputs, max_new_tokens=5, do_sample=False)

print(f"\nPEFT model output:   {tokenizer.decode(peft_out[0], skip_special_tokens=True)}")
print(f"Merged model output: {tokenizer.decode(merged_out[0], skip_special_tokens=True)}")
print(f"Outputs match: {torch.equal(peft_out, merged_out)}")